<a href="https://colab.research.google.com/github/janithcyapa/DHCA-Framework/blob/main/2.Thermal_Zone_Modeling_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validating Thermal ZOne Model in Energy Plus


### Setup Environement

In [1]:
!pip uninstall -y energy-plus-utility

In [2]:
!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✅ Installed 'energy-plus-utility' version: 0.2.2+5


In [3]:
from eplus import prepare_colab_eplus
prepare_colab_eplus()

In [4]:
!pip install control

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.3/578.3 kB 12.3 MB/s eta 0:00:00


## Setup Model

In [5]:
from eplus.core import EPlusUtil
import types
import datetime

import pandas as pd
import numpy as np
import control as ct
import traceback


In [6]:
# @title Setup Model
OUT_DIR = "/simulation/eplus_out"
url_idf="https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneAirCooled.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

# Initialize Utility
sim = EPlusUtil(verbose=1, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
sim.set_model_from_url(url_idf, url_epw)

sim.ensure_output_sqlite()
sim.prepare_run_with_co2(
    outdoor_co2_ppm=420.0,
    wipe_outputs=True,
    activate=True,
    reset=True
)
sim.run_dry_run(include_ems_edd=False,reset=True,design_day=True)

# catalog = sim.api_catalog_df()
# mo.ui.table(catalog)
# mo.ui.table(catalog['VARIABLES'])
# sim.list_available_variables()

EnergyPlus state has been reset.
EnergyPlus state has been reset.
Model set: IDF='/simulation/eplus_out/5ZoneAirCooled.idf', EPW='/simulation/eplus_out/LKA_Colombo-Katunayake.434500_SWERA.epw', OUT_DIR='/simulation/eplus_out'
EnergyPlus state has been reset.
EnergyPlus state has been reset.
EnergyPlus state has been reset.


0

## Setup Simulator

In [7]:
# @title Setup Data Logger
# Request the variables to construct State Vector (x_i) and Disturbances (d_i)
specs = [
    # Zone States (x_i)
    {"name": "Zone Mean Air Temperature", "key": "*"},       # T_in,i
    {"name": "Zone Mean Radiant Temperature", "key": "*"},   # T_m,i (Thermal Mass proxy)
    {"name": "Zone Mean Air Humidity Ratio", "key": "*"},    # W_in,i
    {"name": "Zone Air Relative Humidity", "key": "*"},      # W_in,i (%)
    {"name": "Zone Air CO2 Concentration", "key": "*"},      # ppm

    # Time-Varying Parameters (p_i)
    {"name": "Zone People Occupant Count", "key": "*"},      # No. of People
    {"name": "Zone Electric Equipment Total Heating Rate", "key": "*"}, # Watts

    # External Environment Conditions (x_out)
    {"name": "Site Outdoor Air Drybulb Temperature", "key": "*"}, # T_out
    {"name": "Site Outdoor Air Humidity Ratio", "key": "*"},      # W_out
    {"name": "Site Outdoor Air Relative Humidity", "key": "*"},   # W_in,i (%)
    {"name": "Schedule Value", "key": "CO2-Outdoor-Actuated"},     # ppm

    # Control Inputs - VAV Box Volumetric Flow Rate (m3/s)
    {"name": "System Node Current Density Volume Flow Rate", "key": "*"},

    # AHU Supply Parameters (S)
    {"name": "System Node Temperature", "key": "*"},       # T_s
    {"name": "System Node Humidity Ratio", "key": "*"},    # W_s
    {"name": "System Node CO2 Concentration", "key": "*"}, # C_s
]
sim.ensure_output_variables(specs, activate=True)

sim.collected_data = []
sim.current_state = {}

def state_logger(self, state):
    """Extracts sensor data, updates the current snapshot, and logs history."""
    if not self.exchange.api_data_fully_ready(state):
        return

    # 1. Get Simulation Time Details
    day = self.exchange.day_of_year(state)
    time_now = self.exchange.current_time(state)
    total_minutes = int(time_now * 60)
    hours, mins = divmod(total_minutes, 60)

    row = {
        "timestamp": f"Day {day:03d} {hours:02d}:{mins:02d}",
        "day": day,
        "hour": hours,
        "minute": mins,
        "time_decimal": time_now
    }

    # 2. Extract Outdoor and Supply Data
    t_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Drybulb Temperature", "Environment")
    w_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Humidity Ratio", "Environment")
    rh_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Relative Humidity", "Environment")
    co2_out_h = self.exchange.get_variable_handle(state, "Schedule Value", "CO2-Outdoor-Actuated")

    row["T_out"] = self.exchange.get_variable_value(state, t_out_h)
    row["W_out"] = self.exchange.get_variable_value(state, w_out_h)
    row["RH_out_%"] = self.exchange.get_variable_value(state, rh_out_h)
    row["CO2_out"] = self.exchange.get_variable_value(state, co2_out_h)


    t_s_h = self.exchange.get_variable_handle(state, "System Node Temperature", "VAV Sys 1 Outlet Node")
    w_s_h = self.exchange.get_variable_handle(state, "System Node Humidity Ratio", "VAV Sys 1 Outlet Node")
    c_s_h = self.exchange.get_variable_handle(state, "System Node CO2 Concentration", "VAV Sys 1 Outlet Node")

    row["T_s"] = self.exchange.get_variable_value(state, t_s_h)
    row["W_s"] = self.exchange.get_variable_value(state, w_s_h)
    row["C_s"] = self.exchange.get_variable_value(state, c_s_h)

    # 3. Extract Internal States for All Zones
    zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]
    for zone in zones:
        t_in_h = self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone)
        t_m_h = self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone)
        w_in_h = self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone)
        rh_in_h = self.exchange.get_variable_handle(state, "Zone Air Relative Humidity", zone)
        co2_in_h = self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone)
        occ_in_h = self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone)
        q_eq_h = self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone)
        v_dot_h = self.exchange.get_variable_handle(state, "System Node Current Density Volume Flow Rate", f"{zone} In Node")

        row[f"{zone}_T_in"] = self.exchange.get_variable_value(state, t_in_h)
        row[f"{zone}_T_m"] = self.exchange.get_variable_value(state, t_m_h)
        row[f"{zone}_W_in"] = self.exchange.get_variable_value(state, w_in_h)
        row[f"{zone}_RH_%"] = self.exchange.get_variable_value(state, rh_in_h)
        row[f"{zone}_CO2_in"] = self.exchange.get_variable_value(state, co2_in_h)
        row[f"{zone}_Occ"] = self.exchange.get_variable_value(state, occ_in_h)
        row[f"{zone}_Q_equip"] = self.exchange.get_variable_value(state, q_eq_h)
        row[f"{zone}_V_dot"] = self.exchange.get_variable_value(state, v_dot_h)

    # 4. Update the current snapshot AND append to historical log
    self.current_state = row
    self.collected_data.append(row)

sim.state_logger = types.MethodType(state_logger, sim)
sim.register_handlers(
    "begin", [
        {"method_name": "state_logger"},
        {"method_name": "occupancy_handler", "kwargs": {"lam": 3.0, "min": 0, "max": 5, "seed": 4 } },
        {"method_name": "co2_set_outdoor_ppm", "kwargs": { "value_ppm": 420.0, "log_every_minutes": 60 } }
    ]
)

print(f"Handlers on 'begin' hook: {sim.list_handlers("begin")}")

EnergyPlus state has been reset.
Handlers on 'begin' hook: ['state_logger', 'occupancy_handler', 'co2_set_outdoor_ppm']


In [8]:
def zone_1_model(self, state):


    try:
        if not self.exchange.api_data_fully_ready(state):
            return
        zone_id = "SPACE1-1"

        # Time Stamping
        day = self.exchange.day_of_year(state)
        time = self.exchange.current_time(state)
        base_date = datetime.datetime(2026, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time * 3600)))
        abs_time = (day * 24.0) + time

        print("Zone :"+zone_id+" Time :"+str(base_date.strftime("%Y-%m-%d %H:%M:%S")))

        # Check If Zones is initialized
        if not hasattr(self, 'zones'):
            self.zones = {}

        # --- Initiallize Zone ---
        if zone_id not in self.zones:

            # --- Fetch Parameters and Initialize ---
            raw_params = self.get_zone_thermal_parameters()[zone_id]
            handles = {
                "T_in": self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone_id),
                "T_m": self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone_id),
                "W_in": self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone_id),
                "CO2_in": self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone_id),
                "N_occ": self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone_id),
                "T_out": self.exchange.get_variable_handle(state, "Site Outdoor Air Drybulb Temperature", "Environment"),
                "Q_equip": self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone_id),
                "V_dot": self.exchange.get_variable_handle(state, "System Node Current Density Volume Flow Rate", f"{zone_id} In Node"),
                "T_s": self.exchange.get_variable_handle(state, "System Node Temperature", "VAV Sys 1 Outlet Node"),
                "W_s": self.exchange.get_variable_handle(state, "System Node Humidity Ratio", "VAV Sys 1 Outlet Node"),
                "C_s": self.exchange.get_variable_handle(state, "System Node CO2 Concentration", "VAV Sys 1 Outlet Node"),
            }

            inv_R_env_ext = 0.0
            R_env_gnd = None
            adj_zones = []

            for b in raw_params["boundaries"]:
              target = b["target"]
              r_abs = float(b["R_absolute_K_W"])
              if target == "Ground": R_env_gnd = r_abs
              elif target == "Environment" or b["boundary_condition"] == "outdoors": inv_R_env_ext += (1.0 / r_abs)
              else: adj_zones.append({ "zone": target, "R_env": r_abs, "handle_T_in": self.exchange.get_variable_handle( state, "Zone Mean Air Temperature", target )})
            R_env_ext = 1.0 / inv_R_env_ext if inv_R_env_ext > 0 else float('inf')

            # --- Store in Namespace ---
            self.zones[zone_id] = types.SimpleNamespace(
                last_time=abs_time,
                V_room=float(raw_params['V_room']),
                M_air=float(raw_params['M_air']),
                C_air=float(raw_params['C_air']),
                C_mass=float(raw_params['C_mass']),
                R_int=float(raw_params['R_int']),
                R_env_gnd=R_env_gnd,
                R_env_ext=R_env_ext,
                adj_zones=adj_zones,
                handles=handles,
                log=[]
            )

            # --- Testing Print ---
            z = self.zones[zone_id]
            print(f"\n--- [{zone_id}] Param Extraction ---")
            print(f"Physical: V={z.V_room}, M_air={z.M_air}")
            print(f"R_ground: {z.R_env_gnd}")
            print(f"R_env (External Merged): {z.R_env_ext:.6f}")
            print(f"Adjacent Zones Array: {z.adj_zones}")
            print("-----------------------------------\n")
            print(f"[{zone_id}] Handles initialized. Neighbors: {[az['zone'] for az in adj_zones]}")

        else:
            self.zones[zone_id].last_time = abs_time

        row = {
            "timestamp": base_date.strftime("%Y-%m-%d %H:%M:%S"),
        }



    except Exception as e:
        print(f"\n--- Python Exception in zone_1_model ---")
        print(f"Error: {e}")
        traceback.print_exc()
        print("----------------------------------------\n")

sim.zone_1_model = types.MethodType(zone_1_model, sim)
sim.register_handlers("begin", [{"method_name": "zone_1_model"}])

['state_logger', 'occupancy_handler', 'co2_set_outdoor_ppm', 'zone_1_model']

In [9]:
# @title Run Simulation
# Set Simulation Time
sim.set_simulation_params(
    start=(1, 1),
    end=(1, 1),
    timestep_per_hour = 1, # 4 (every 15 minutes) or 6 (every 10 minutes).
    start_day_of_week="Sunday",
)

print("Starting EnergyPlus Uncontrolled Simulation...")
res = sim.run_annual()

if(res == 0):
    print("Simulation Complete! Converting data to Pandas...")
    # Create the DataFrame
    SimulationData = pd.DataFrame(sim.collected_data)
    cols = ['timestamp'] + [c for c in SimulationData.columns if c != 'timestamp']
    SimulationData = SimulationData[cols]
    print("Done.")

if(res == 1):
    err_path = OUT_DIR / "eplusout.err"
    if err_path.exists():
        print("--- EnergyPlus Error Log ---")
        with open(err_path, 'r') as f:
            # Print the last 4000 characters to catch the fatal errors at the end
            print(f.read()[-4000:])
    else:
        print(f"Could not find the error file at: {err_path}")

EnergyPlus state has been reset.
Starting EnergyPlus Uncontrolled Simulation...
Deleted output file: /simulation/eplus_out/eplusout.err
Deleted output file: /simulation/eplus_out/eplusout.audit
EnergyPlus state has been reset.
[OCC] Note: no People object matched zone 'PLENUM-1' (will be ignored).
[occ-counter] Resolved 5 People handles across 5 zones.
[co2-outdoor] set=420.0 ppm  read-back=420.0 ppm
Zone :SPACE1-1 Time :2026-01-01 01:00:00

--- [SPACE1-1] Param Extraction ---
Physical: V=239.247360229, M_air=288.05
R_ground: 0.0023
R_env (External Merged): 0.004300
Adjacent Zones Array: [{'zone': 'PLENUM-1', 'R_env': 0.0081, 'handle_T_in': 23}, {'zone': 'SPACE2-1', 'R_env': 0.0199, 'handle_T_in': 27}, {'zone': 'SPACE4-1', 'R_env': 0.0199, 'handle_T_in': 31}, {'zone': 'SPACE5-1', 'R_env': 0.0045, 'handle_T_in': 33}]
-----------------------------------

[SPACE1-1] Handles initialized. Neighbors: ['PLENUM-1', 'SPACE2-1', 'SPACE4-1', 'SPACE5-1']
Zone :SPACE1-1 Time :2026-01-01 02:00:00
Zo

In [ ]:
# @title
SimulationData